# regimekit tutorial: from data to a regime forecast

This notebook walks through the Python API (`regimekit.Project`) end to end: validate a config, fetch and prepare data, fit a 3-state hidden semi-Markov model, decode regimes, forecast and backtest.

**Data.** To run offline (and in CI) the notebook uses a *synthetic look-alike* of an equity index and two covariates, simulated from the example model by `examples/synthetic.py`. For real data, point `Project.from_yaml` at `examples/configs/dax_3state_gbm_nb.yaml` (or the S&P 500 variant); it fetches `^GDAXI`, `VIXCLS` and `T10Y2Y` (a `FRED_API_KEY` is needed for the FRED series).

In [ ]:
import sys
import tempfile
from pathlib import Path

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "examples" else Path.cwd()
sys.path.insert(0, str(ROOT))

from examples.synthetic import write_offline_project  # noqa: E402
from regimekit import Project  # noqa: E402

work = Path(tempfile.mkdtemp(prefix="regimekit_tutorial_"))
config_path = write_offline_project(work, ROOT / "examples" / "configs" / "dax_3state_gbm_nb.yaml")
project = Project.from_yaml(config_path)
print(f"workdir: {work}")

## 1. Validate the configuration

`validate()` summarizes the model the config describes.

In [ ]:
for line in project.validate():
    print(line)

## 2. Fetch and prepare

`fetch()` reads the provider through the snapshot cache and returns the `PreparedDataset`: log-returns, step sizes `dt`, lagged and standardized covariates. The data-quality report stays on `project.last_quality`.

In [ ]:
data = project.fetch()
print(f"{data.times.shape[0]} rows, snapshot {data.snapshot_ids[0][:12]}")
print("covariates:", data.prep_stats.cov_names)

## 3. Fit

`fit()` runs multi-start MAP estimation. The regimes are ordered by volatility (`sigma` increasing), so regime 0 is the calm one.

In [ ]:
fit = project.fit(data)
print(f"loglik {fit.loglik:.1f}, converged {fit.converged}, {len(fit.restarts)} restarts")
print("sigma (annualized):", np.round(np.asarray(fit.params["emission.sigma"]), 3))

## 4. Decode

`decode()` returns the filtered (real-time) and smoothed regime probabilities, the Viterbi path and information criteria.

In [ ]:
post = project.decode(fit, data)
print(f"AIC {post.aic:.1f}  BIC {post.bic:.1f}  ICL {post.icl:.1f}")
times = pd.to_datetime(post.times)
filtered = np.asarray(post.filtered)
fig, ax = plt.subplots(figsize=(9, 3))
ax.stackplot(times, filtered.T, labels=[f"regime {i}" for i in range(filtered.shape[1])])
ax.set_ylim(0, 1)
ax.set_title("Filtered regime probabilities")
ax.legend(loc="upper right", ncols=filtered.shape[1])
fig.tight_layout()
print(f"{len(post.segments)} Viterbi segments")

## 5. Forecast

`forecast()` simulates future regimes and returns from the end of the data. The result holds regime probabilities per step and quantiles of the cumulative log-return (the fan).

In [ ]:
fc = project.forecast(fit, data, horizon=10)
steps = np.arange(1, fc.cum_quantiles.shape[0] + 1)
fan = np.asarray(fc.cum_quantiles)[:, :, 0]
fig, ax = plt.subplots(figsize=(9, 3))
for q in range(fan.shape[1]):
    ax.plot(steps, fan[:, q], label=f"q{fc.quantile_levels[q]:.2f}")
ax.set_xlabel("steps ahead")
ax.set_title("Cumulative log-return fan")
ax.legend(ncols=fan.shape[1])
fig.tight_layout()
pd.DataFrame(np.asarray(fc.regime_probs), index=steps).round(3).head()

## 6. Backtest

`backtest()` refits on expanding windows and scores each fold out of sample: predictive log score, CRPS, PIT coverage, and a regime strategy with costs.

In [ ]:
bt = project.backtest()
print(f"{len(bt.folds)} folds")
bt.summary.head(12)

In [ ]:
strategy = bt.strategy
fig, ax = plt.subplots(figsize=(9, 3))
ax.plot(pd.to_datetime(strategy["time"]), strategy["equity"])
ax.set_title("Out-of-sample equity of the regime long/flat strategy")
fig.tight_layout()

## Where to go next

- `regimekit` CLI mirrors every call above (`regimekit fit config.yaml`, ...).
- `Project.compare([cfg_a, cfg_b])` backtests several configs on one data snapshot.
- See the guide pages on configuration, forecasting, backtesting and the model bundle.